In [1]:
%%writefile kernel.h
#ifndef KERNEL_H
#define KERNEL_H

#define LOWER_BOUND -5.12f
#define UPPER_BOUND 5.12f

#ifdef __cplusplus
extern "C" {
#endif

float cuda_pso(float* h_pop, int pop_size, int dim, int max_iter, int func_id);
float cpu_pso(float* h_pop, float* h_vel, float* h_pbest, float* h_pbest_fit, float* h_gbest, int pop_size, int dim, int max_iter, int func_id);
float getRandom();

#ifdef __cplusplus
}
#endif

#endif

Writing kernel.h


In [2]:
%%writefile kernel.cu
#include "kernel.h"
#include <cuda_runtime.h>
#include <device_launch_parameters.h>
#include <cmath>
#include <cfloat>

#ifndef M_PI
#define M_PI 3.14159265358979323846f
#endif

// Évaluation GPU des 5 fonctions avec biais (PDF Slide 6)
__device__ float eval_function_gpu(const float* x, int dim, int func_id) {
    float sum = 0.0f;
    
    if (func_id == 0) { 
        // 1. Levy (f_bias = -180.0f)
        float f_bias = -180.0f;
        for (int i = 0; i < dim; i++) {
            float w = 1.0f + (x[i] - 1.0f) / 4.0f;
            sum += (w - 1.0f) * (w - 1.0f) * (1.0f + 10.0f * sinf(M_PI * w + 1.0f));
        }
        return sum + f_bias;
        
    } else if (func_id == 1) { 
        // 2. Rastrigin (f_bias = -330.0f)
        float f_bias = -330.0f;
        for (int i = 0; i < dim; i++) {
            sum += (x[i] * x[i] - 10.0f * cosf(2.0f * M_PI * x[i]) + 10.0f);
        }
        return sum + f_bias;
        
    } else if (func_id == 2) { 
        // 3. Rosenbrock (f_bias = 390.0f)
        float f_bias = 390.0f;
        for (int i = 0; i < dim - 1; i++) {
            float t1 = x[i + 1] - x[i] * x[i];
            float t2 = 1.0f - x[i];
            sum += 100.0f * t1 * t1 + t2 * t2;
        }
        return sum + f_bias;
        
    } else if (func_id == 3) { 
        // 4. Griewank (f_bias = -180.0f)
        float prod = 1.0f;
        float f_bias = -180.0f;
        for (int i = 0; i < dim; i++) {
            sum += (x[i] * x[i]) / 4000.0f;
            prod *= cosf(x[i] / sqrtf(static_cast<float>(i + 1)));
        }
        return sum - prod + 1.0f + f_bias;
        
    } else { 
        // 5. Sphere (f_bias = -450.0f)
        float f_bias = -450.0f;
        for (int i = 0; i < dim; i++) {
            sum += x[i] * x[i];
        }
        return sum + f_bias;
    }
}

// Kernel CUDA PSO : mise à jour vitesse, position et pbest
__global__ void pso_kernel(float* pop, float* vel, float* pbest, float* pbest_fit, const float* gbest, int pop_size, int dim, int func_id, float w, float c1, float c2) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= pop_size) return;

    for (int d = 0; d < dim; d++) {
        int pos_idx = idx * dim + d;
        float r1 = 0.5f; // Générateur simplifié pour démo
        float r2 = 0.5f;

        // v_i = w*v_i + c1*r1*(pbest_i - x_i) + c2*r2*(gbest - x_i)
        vel[pos_idx] = w * vel[pos_idx] + c1 * r1 * (pbest[pos_idx] - pop[pos_idx]) + c2 * r2 * (gbest[d] - pop[pos_idx]);
        pop[pos_idx] += vel[pos_idx];
    }

    float current_fit = eval_function_gpu(&pop[idx * dim], dim, func_id);

    if (current_fit < pbest_fit[idx]) {
        pbest_fit[idx] = current_fit;
        for (int d = 0; d < dim; d++) {
            pbest[idx * dim + d] = pop[idx * dim + d];
        }
    }
}

extern "C" float cuda_pso(float* h_pop, int pop_size, int dim, int max_iter, int func_id) {
    float *d_pop, *d_vel, *d_pbest, *d_pbest_fit, *d_gbest;
    size_t size_pop = pop_size * dim * sizeof(float);
    size_t size_fit = pop_size * sizeof(float);
    size_t size_gbest = dim * sizeof(float);

    cudaMalloc((void**)&d_pop, size_pop);
    cudaMalloc((void**)&d_vel, size_pop);
    cudaMalloc((void**)&d_pbest, size_pop);
    cudaMalloc((void**)&d_pbest_fit, size_fit);
    cudaMalloc((void**)&d_gbest, size_gbest);

    cudaMemset(d_vel, 0, size_pop);
    cudaMemcpy(d_pop, h_pop, size_pop, cudaMemcpyHostToDevice);
    cudaMemcpy(d_pbest, h_pop, size_pop, cudaMemcpyHostToDevice);

    // Initialisation pbest_fit
    float* h_init_fit = new float[pop_size];
    for (int i = 0; i < pop_size; i++) h_init_fit[i] = FLT_MAX;
    cudaMemcpy(d_pbest_fit, h_init_fit, size_fit, cudaMemcpyHostToDevice);
    delete[] h_init_fit;

    float* h_gbest = new float[dim]();
    cudaMemcpy(d_gbest, h_gbest, size_gbest, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (pop_size + threadsPerBlock - 1) / threadsPerBlock;

    for (int iter = 0; iter < max_iter; iter++) {
        pso_kernel<<<blocksPerGrid, threadsPerBlock>>>(d_pop, d_vel, d_pbest, d_pbest_fit, d_gbest, pop_size, dim, func_id, 0.7f, 1.4f, 1.4f);
        cudaDeviceSynchronize();
    }

    float* h_pbest_fit = new float[pop_size];
    cudaMemcpy(h_pbest_fit, d_pbest_fit, size_fit, cudaMemcpyDeviceToHost);

    float best_fit = FLT_MAX;
    for (int i = 0; i < pop_size; i++) {
        if (h_pbest_fit[i] < best_fit) best_fit = h_pbest_fit[i];
    }

    delete[] h_pbest_fit;
    delete[] h_gbest;
    cudaFree(d_pop);
    cudaFree(d_vel);
    cudaFree(d_pbest);
    cudaFree(d_pbest_fit);
    cudaFree(d_gbest);

    return best_fit;
}

Writing kernel.cu


In [3]:
%%writefile kernel.cpp
#include "kernel.h"
#include <cstdlib>
#include <cmath>
#include <cfloat>

#ifndef M_PI
#define M_PI 3.14159265358979323846f
#endif

extern "C" float getRandom() {
    return (float)rand() / RAND_MAX;
}

// Évaluation CPU des 5 fonctions
float eval_function_cpu(const float* x, int dim, int func_id) {
    float sum = 0.0f;
    
    if (func_id == 0) { 
        float f_bias = -180.0f;
        for (int i = 0; i < dim; i++) {
            float w = 1.0f + (x[i] - 1.0f) / 4.0f;
            sum += (w - 1.0f) * (w - 1.0f) * (1.0f + 10.0f * sinf(M_PI * w + 1.0f));
        }
        return sum + f_bias;
        
    } else if (func_id == 1) { 
        float f_bias = -330.0f;
        for (int i = 0; i < dim; i++) {
            sum += (x[i] * x[i] - 10.0f * cosf(2.0f * M_PI * x[i]) + 10.0f);
        }
        return sum + f_bias;
        
    } else if (func_id == 2) { 
        float f_bias = 390.0f;
        for (int i = 0; i < dim - 1; i++) {
            float t1 = x[i + 1] - x[i] * x[i];
            float t2 = 1.0f - x[i];
            sum += 100.0f * t1 * t1 + t2 * t2;
        }
        return sum + f_bias;
        
    } else if (func_id == 3) { 
        float prod = 1.0f;
        float f_bias = -180.0f;
        for (int i = 0; i < dim; i++) {
            sum += (x[i] * x[i]) / 4000.0f;
            prod *= cosf(x[i] / sqrtf(static_cast<float>(i + 1)));
        }
        return sum - prod + 1.0f + f_bias;
        
    } else { 
        float f_bias = -450.0f;
        for (int i = 0; i < dim; i++) {
            sum += x[i] * x[i];
        }
        return sum + f_bias;
    }
}

extern "C" float cpu_pso(float* h_pop, float* h_vel, float* h_pbest, float* h_pbest_fit, float* h_gbest, int pop_size, int dim, int max_iter, int func_id) {
    float global_best_fit = FLT_MAX;

    for (int iter = 0; iter < max_iter; iter++) {
        for (int i = 0; i < pop_size; i++) {
            for (int d = 0; d < dim; d++) {
                int pos_idx = i * dim + d;
                float r1 = 0.5f;
                float r2 = 0.5f;
                
                h_vel[pos_idx] = 0.7f * h_vel[pos_idx] + 1.4f * r1 * (h_pbest[pos_idx] - h_pop[pos_idx]) + 1.4f * r2 * (h_gbest[d] - h_pop[pos_idx]);
                h_pop[pos_idx] += h_vel[pos_idx];
            }

            float current_fit = eval_function_cpu(&h_pop[i * dim], dim, func_id);

            if (current_fit < h_pbest_fit[i]) {
                h_pbest_fit[i] = current_fit;
                for (int d = 0; d < dim; d++) h_pbest[i * dim + d] = h_pop[i * dim + d];

                if (current_fit < global_best_fit) {
                    global_best_fit = current_fit;
                    for (int d = 0; d < dim; d++) h_gbest[d] = h_pop[i * dim + d];
                }
            }
        }
    }
    return global_best_fit;
}

Writing kernel.cpp


In [4]:
%%writefile main.cpp
#include "kernel.h"
#include <chrono>
#include <iostream>
#include <vector>
#include <cstdlib>
#include <cfloat>

int main(int argc, char* argv[]) {
    int func_id = (argc > 1) ? atoi(argv[1]) : 0;
    int dim     = (argc > 2) ? atoi(argv[2]) : 10;
    int pop_size= (argc > 3) ? atoi(argv[3]) : 50;

    int max_evals = 10000 * dim;
    int max_iter  = max_evals / pop_size;

    std::vector<float> h_pop(pop_size * dim);
    std::vector<float> h_vel(pop_size * dim, 0.0f);
    std::vector<float> h_pbest(pop_size * dim);
    std::vector<float> h_pbest_fit(pop_size, FLT_MAX);
    std::vector<float> h_gbest(dim, 0.0f);

    srand(1234);
    for (size_t i = 0; i < h_pop.size(); i++) {
        h_pop[i] = LOWER_BOUND + getRandom() * (UPPER_BOUND - LOWER_BOUND);
        h_pbest[i] = h_pop[i];
    }

    // Mesure du temps CPU
    auto start_cpu = std::chrono::high_resolution_clock::now();
    float cpu_fit = cpu_pso(h_pop.data(), h_vel.data(), h_pbest.data(), h_pbest_fit.data(), h_gbest.data(), pop_size, dim, max_iter, func_id);
    auto end_cpu = std::chrono::high_resolution_clock::now();
    double cpu_time = std::chrono::duration<double, std::milli>(end_cpu - start_cpu).count();

    // Mesure du temps GPU
    auto start_gpu = std::chrono::high_resolution_clock::now();
    float gpu_fit = cuda_pso(h_pop.data(), pop_size, dim, max_iter, func_id);
    auto end_gpu = std::chrono::high_resolution_clock::now();
    double gpu_time = std::chrono::duration<double, std::milli>(end_gpu - start_gpu).count();

    double speedup = cpu_time / gpu_time;

    std::cout << func_id << "," << dim << "," << max_iter << "," << pop_size << ","
              << cpu_time << "," << gpu_time << "," << speedup << ","
              << cpu_fit << "," << gpu_fit << std::endl;

    return 0;
}

Writing main.cpp


In [5]:
!nvcc -O3 main.cpp kernel.cpp kernel.cu -o pso_app

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [6]:
import subprocess
import pandas as pd
from IPython.display import FileLink

fonctions = {0: 'Levy', 1: 'Rastrigin', 2: 'Rosenbrock', 3: 'Griewank', 4: 'Sphere'}
dimensions = [10, 50, 100]
populations = [50, 100, 500]

resultats = []
for func_id, func_name in fonctions.items():
    for dim in dimensions:
        for pop in populations:
            cmd = f"./pso_app {func_id} {dim} {pop}"
            output = subprocess.check_output(cmd, shell=True).decode('utf-8').strip()
            f_id, d, g, p, t_cpu, t_gpu, sp, fit_cpu, fit_gpu = output.split(',')
            resultats.append({
                'Fonction': func_name, 'Dimension': int(d), 'Generations': int(g), 'Population': int(p),
                'Temps CPU (ms)': round(float(t_cpu), 2), 'Temps GPU (ms)': round(float(t_gpu), 2),
                'Speedup': f"{float(sp):.2f}x", 'Fitness CPU': f"{float(fit_cpu):.3e}", 'Fitness GPU': f"{float(fit_gpu):.3e}"
            })

df = pd.DataFrame(resultats)
df.to_excel('tableau_final_PSO.xlsx', index=False)
display(df)
FileLink('tableau_final_PSO.xlsx')

,Fonction,Dimension,Generations,Population,Temps CPU (ms),Temps GPU (ms),Speedup,Fitness CPU,Fitness GPU
0,Levy,10,2000,50,13.06,712.68,0.02x,-1.904e+02,-1.815e+02
1,Levy,10,1000,100,13.79,360.18,0.04x,-1.894e+02,-1.814e+02
2,Levy,10,200,500,12.75,337.78,0.04x,-1.894e+02,-1.814e+02
3,Levy,50,10000,50,295.44,728.25,0.41x,-1.992e+02,-1.844e+02
4,Levy,50,5000,100,289.62,450.77,0.64x,-2.066e+02,-1.851e+02
5,Levy,50,1000,500,325.48,352.28,0.92x,-2.164e+02,-1.864e+02
6,Levy,100,20000,50,1153.36,1417.27,0.81x,-1.946e+02,-1.882e+02
7,Levy,100,10000,100,1148.30,963.91,1.19x,-1.985e+02,-1.883e+02
8,Levy,100,2000,500,1228.04,584.57,2.10x,-2.241e+02,-1.898e+02
9,Rastrigin,10,2000,50,9.59,297.24,0.03x,-3.244e+02,-3.300e+02


/kaggle/working/tableau_final_PSO.xlsx